# 영상분류 해커톤 — 스타터 노트북
## 컴퓨터비전 부트캠프 2일차 (8/4) 3·4차시

---

### 팀 정보 — 먼저 채워 주세요

| 항목 | 내용 |
|---|---|
| 팀명 | |
| 팀원 | |

---

### 규칙

**허용** — torchvision 사전학습 모델, 모든 증강 기법, 하이퍼파라미터 자유, 앙상블, 검색·문서 참고

**금지** — test 셋으로 모델 선택하기, 외부 CIFAR-10 정답 파일, 다른 팀 코드 복사,
train/val 분할 비율 변경, 실험 기록 없이 결과만 제출

> **가장 중요한 규칙** — 모델 선택과 튜닝은 validation 으로만. test 정확도는 **마지막에 딱 한 번.**

### 평가 기준

| 항목 | 배점 | 보는 것 |
|---|---|---|
| 분류 성능 | 30 | test 정확도 |
| 실험 설계 | 30 | 한 번에 하나씩 바꿨는가 (실험 기록표) |
| 코드 완성도 | 20 | 처음부터 끝까지 오류 없이 실행되는가 |
| 결과 분석 | 20 | 왜 좋아졌는지 설명할 수 있는가 |

### 제출물 (13:20까지)

1. 이 노트북 (처음부터 끝까지 실행되는 상태) — 파일명 `팀명_해커톤.ipynb`
2. 섹션 6의 실험 기록표
3. 섹션 8의 결과 요약 5줄

### 시간 배분 권장

| 시각 | 할 일 |
|---|---|
| 11:30 ~ 11:45 | 안내 · 팀 구성 · 노트북 열기 |
| 11:45 ~ 12:10 | 섹션 0~5 실행해 baseline 점수 확인 |
| 12:10 ~ 12:40 | 개선 실험 1차 (증강 → 전이학습) |
| 12:40 ~ 13:05 | 개선 실험 2차 · 최적 설정 확정 |
| **13:05 ~ 13:20** | **학습 중단. 기록표와 요약 작성** |

---
## 0. 환경 준비

그대로 실행하세요. 수정할 필요 없습니다.

In [ ]:
import random, time, copy
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset, random_split
from torchvision import datasets, transforms, models

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("장치 :", device)
if device.type != "cuda":
    print("⚠️  런타임 → 런타임 유형 변경 → T4 GPU 로 바꿔 주세요.")

CIFAR10_MEAN  = (0.4914, 0.4822, 0.4465)
CIFAR10_STD   = (0.2470, 0.2435, 0.2616)
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)
CLASSES = ["airplane", "automobile", "bird", "cat", "deer",
           "dog", "frog", "horse", "ship", "truck"]
plt.rcParams["figure.dpi"] = 110

### 그래프에 한글 쓰기 (Colab)

Colab 의 matplotlib 에는 한글 폰트가 없어 그래프 안의 한글이 네모로 깨집니다.
아래 셀을 한 번 실행해 두면 해결됩니다. (약 20초)

In [ ]:
# Colab 한글 폰트 설정 — 실패해도 실습에는 지장이 없습니다
try:
    import subprocess, matplotlib.font_manager as fm
    subprocess.run("apt-get install -y -qq fonts-nanum", shell=True, check=True)
    fm.fontManager.addfont("/usr/share/fonts/truetype/nanum/NanumGothic.ttf")
    plt.rcParams["font.family"] = "NanumGothic"
    print("한글 폰트 설정 완료")
except Exception as e:
    print("한글 폰트 설정을 건너뜁니다 :", e)
plt.rcParams["axes.unicode_minus"] = False

---
## 1. 설정 — **여기를 바꾼다**

실험할 때 건드리는 곳은 이 딕셔너리 하나입니다.
값을 바꾸고 섹션 5의 `run_experiment()` 를 다시 실행하면 한 번의 실험이 끝납니다.

In [ ]:
CONFIG = {
    "name":        "baseline",      # 실험 이름 — 기록표에 그대로 들어간다

    # --- 모델 ---
    "model":       "simplecnn",     # simplecnn | resnet18 | resnet50 | efficientnet_b0
    "pretrained":  False,           # True 면 ImageNet 사전학습 가중치 사용
    "freeze":      False,           # True 면 backbone 고정 (Feature Extractor)

    # --- 데이터 ---
    "img_size":    32,              # 32 | 112 | 224  (사전학습 모델은 112 이상 권장)
    "augment":     [],              # "crop", "flip", "jitter", "erasing" 중 골라 담기
    "batch_size":  128,             # 메모리 부족(OOM)이면 64 또는 32

    # --- 학습 ---
    "epochs":      10,
    "lr":          1e-3,            # 사전학습 fine-tuning 이면 1e-4 권장
    "head_lr":     None,            # 새 분류층만 다른 lr 을 주고 싶을 때 (예: 1e-3)
    "weight_decay": 1e-2,
    "scheduler":   None,            # None | "cosine" | "step"

    "note":        "시작점",         # 기록표 메모
}

print(CONFIG)

### 설정값 빠른 안내

| 하고 싶은 것 | 바꿀 값 |
|---|---|
| 증강 추가 | `"augment": ["crop", "flip"]` |
| ResNet18 Feature Extractor | `"model": "resnet18", "pretrained": True, "freeze": True, "img_size": 112` |
| ResNet18 Fine-tuning | `"model": "resnet18", "pretrained": True, "freeze": False, "img_size": 112, "lr": 1e-4, "head_lr": 1e-3` |
| 스케줄러 추가 | `"scheduler": "cosine"` |
| 더 오래 학습 | `"epochs": 20` |

> `img_size` 를 키우면 학습이 크게 느려집니다. 224를 쓰려면 `batch_size` 를 32로 줄이세요.

---
## 2. transform 만들기 — 여기를 바꿔도 된다

`CONFIG["augment"]` 목록을 읽어 학습용 transform 을 조립합니다.
새로운 증강을 추가하고 싶으면 이 함수에 항목을 더하세요.

In [ ]:
def build_transforms(cfg):
    size = cfg["img_size"]
    mean, std = (IMAGENET_MEAN, IMAGENET_STD) if cfg["pretrained"] \
                else (CIFAR10_MEAN, CIFAR10_STD)

    pre = [transforms.Resize(size)] if size != 32 else []

    aug = []
    if "crop" in cfg["augment"]:
        aug.append(transforms.RandomCrop(size, padding=max(2, size // 8)))
    if "flip" in cfg["augment"]:
        aug.append(transforms.RandomHorizontalFlip())
    if "jitter" in cfg["augment"]:
        aug.append(transforms.ColorJitter(0.2, 0.2, 0.2))

    post = [transforms.ToTensor(), transforms.Normalize(mean, std)]
    if "erasing" in cfg["augment"]:
        post.append(transforms.RandomErasing(p=0.25))   # 텐서에 적용되므로 맨 뒤

    return (transforms.Compose(pre + aug + post),      # 학습용
            transforms.Compose(pre + post))            # 평가용 — 증강 없음


def make_loaders(cfg):
    tf_train, tf_eval = build_transforms(cfg)
    full_train = datasets.CIFAR10("./data", train=True,  download=True, transform=tf_train)
    full_eval  = datasets.CIFAR10("./data", train=True,  download=True, transform=tf_eval)
    test_set   = datasets.CIFAR10("./data", train=False, download=True, transform=tf_eval)

    g = torch.Generator().manual_seed(SEED)
    train_idx, val_idx = random_split(range(50000), [45000, 5000], generator=g)

    bs = cfg["batch_size"]
    return (DataLoader(Subset(full_train, list(train_idx)), batch_size=bs,
                       shuffle=True, num_workers=2, pin_memory=True),
            DataLoader(Subset(full_eval, list(val_idx)), batch_size=256,
                       shuffle=False, num_workers=2, pin_memory=True),
            DataLoader(test_set, batch_size=256, shuffle=False,
                       num_workers=2, pin_memory=True))

---
## 3. 모델 만들기 — 여기를 바꿔도 된다

새로운 backbone 을 쓰고 싶으면 `build_model()` 에 분기를 추가하세요.

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10, img_size=32):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),   nn.BatchNorm2d(32),  nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1),  nn.BatchNorm2d(64),  nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.pool = nn.AdaptiveAvgPool2d(4)        # 입력 크기가 달라져도 4x4 로 맞춰 준다
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.pool(self.features(x)))


def build_model(cfg):
    name = cfg["model"]

    if name == "simplecnn":
        return SimpleCNN(img_size=cfg["img_size"])

    weights = "DEFAULT" if cfg["pretrained"] else None

    if name == "resnet18":
        model = models.resnet18(weights=weights)
        head_name = "fc"
        model.fc = nn.Linear(model.fc.in_features, 10)
    elif name == "resnet50":
        model = models.resnet50(weights=weights)
        head_name = "fc"
        model.fc = nn.Linear(model.fc.in_features, 10)
    elif name == "efficientnet_b0":
        model = models.efficientnet_b0(weights=weights)
        head_name = "classifier"
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, 10)
    else:
        raise ValueError(f"모르는 모델: {name}")

    if cfg["freeze"]:
        for n_, p in model.named_parameters():
            p.requires_grad = n_.startswith(head_name)   # head 만 학습

    model._head_name = head_name
    return model


def make_param_groups(model, cfg):
    """head_lr 이 지정되면 backbone 과 head 에 다른 lr 을 준다."""
    head = getattr(model, "_head_name", None)
    if cfg["head_lr"] is None or head is None:
        return [p for p in model.parameters() if p.requires_grad]
    backbone = [p for n_, p in model.named_parameters()
                if p.requires_grad and not n_.startswith(head)]
    head_ps  = [p for n_, p in model.named_parameters()
                if p.requires_grad and n_.startswith(head)]
    return [{"params": backbone, "lr": cfg["lr"]},
            {"params": head_ps,  "lr": cfg["head_lr"]}]

---
## 4. 학습·평가 함수

수정할 필요 없습니다. 어제 것과 같습니다.

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    tot, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        out = model(images)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        tot += loss.item() * labels.size(0)
        correct += (out.argmax(1) == labels).sum().item()
        total += labels.size(0)
    return tot / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    tot, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        out = model(images)
        tot += criterion(out, labels).item() * labels.size(0)
        correct += (out.argmax(1) == labels).sum().item()
        total += labels.size(0)
    return tot / total, correct / total

---
## 5. 실험 실행

`run_experiment(CONFIG)` 한 줄이면 데이터 로드 → 모델 생성 → 학습 → 검증까지
끝나고, 결과가 `RESULTS` 에 자동으로 쌓입니다.

**같은 셀을 CONFIG 만 바꿔 가며 여러 번 실행하세요.**

In [ ]:
RESULTS = []
BEST = {"acc": 0.0, "model": None, "cfg": None, "loaders": None}


def run_experiment(cfg, verbose=True):
    cfg = copy.deepcopy(cfg)
    print(f"\n=== [{cfg['name']}] 시작 ===")
    print("  ", {k: v for k, v in cfg.items() if k not in ("name", "note")})

    torch.manual_seed(SEED)
    train_loader, val_loader, test_loader = make_loaders(cfg)
    model = build_model(cfg).to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(make_param_groups(model, cfg),
                            lr=cfg["lr"], weight_decay=cfg["weight_decay"])

    scheduler = None
    if cfg["scheduler"] == "cosine":
        scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg["epochs"])
    elif cfg["scheduler"] == "step":
        scheduler = optim.lr_scheduler.StepLR(
            optimizer, step_size=max(1, cfg["epochs"] // 3), gamma=0.1)

    hist = {"train_acc": [], "val_acc": []}
    best_acc, best_state, t0 = 0.0, None, time.time()

    for ep in range(1, cfg["epochs"] + 1):
        tr_loss, tr_acc = train_one_epoch(model, train_loader, criterion, optimizer)
        va_loss, va_acc = evaluate(model, val_loader, criterion)
        if scheduler is not None:
            scheduler.step()
        hist["train_acc"].append(tr_acc); hist["val_acc"].append(va_acc)
        if va_acc > best_acc:
            best_acc = va_acc
            best_state = {k: v.detach().cpu().clone()
                          for k, v in model.state_dict().items()}
        if verbose:
            print(f"  epoch {ep:2d}/{cfg['epochs']} | train {tr_acc:.4f} | val {va_acc:.4f}")

    elapsed = time.time() - t0
    model.load_state_dict(best_state)

    RESULTS.append({"name": cfg["name"], "val_acc": best_acc,
                    "epochs": cfg["epochs"], "sec": elapsed,
                    "note": cfg["note"], "cfg": cfg, "hist": hist})

    if best_acc > BEST["acc"]:
        BEST.update(acc=best_acc, model=model, cfg=cfg,
                    loaders=(train_loader, val_loader, test_loader))
        print(f"  ★ 현재 최고 기록 갱신")

    print(f"=== [{cfg['name']}] 검증 정확도 {best_acc:.4f} ({elapsed:.0f}초) ===")
    return model, hist


# 첫 실행 — baseline
model, hist = run_experiment(CONFIG)

### 다음 실험

아래 셀을 **복사해 가며** 실험을 이어 가세요. `CONFIG` 를 통째로 바꾸지 말고,
바꿀 항목만 덮어쓰면 무엇을 바꿨는지 명확해집니다.

```python
cfg = copy.deepcopy(CONFIG)
cfg.update(name="1. + 증강", augment=["crop", "flip"], note="RandomCrop + Flip")
run_experiment(cfg)
```

In [ ]:
# 실험 1 — 데이터 증강
cfg = copy.deepcopy(CONFIG)
cfg.update(name="1. + 증강",
           augment=["crop", "flip"],
           note="RandomCrop + HorizontalFlip")
run_experiment(cfg)

In [ ]:
# 실험 2 — ResNet18 전이학습 (Fine-tuning)
cfg = copy.deepcopy(CONFIG)
cfg.update(name="2. ResNet18 fine-tuning",
           model="resnet18", pretrained=True, freeze=False,
           img_size=112, batch_size=64,
           augment=["crop", "flip"],
           epochs=5, lr=1e-4, head_lr=1e-3, scheduler="cosine",
           note="backbone lr=1e-4, head lr=1e-3, 112px")
run_experiment(cfg)

In [ ]:
# 실험 3 — 여기부터는 직접 설계하세요
cfg = copy.deepcopy(CONFIG)
cfg.update(name="3. ",
           note="")
# run_experiment(cfg)

---
## 6. 실험 기록표 — **제출물 ①**

실험이 끝날 때마다 이 셀을 실행해 표를 확인하세요.

In [ ]:
def show_results():
    if not RESULTS:
        print("아직 실험이 없습니다."); return
    base = RESULTS[0]["val_acc"]
    print(f"{'#':<3}{'실험':<28}{'val acc':>9}{'변화':>10}{'epoch':>7}{'시간(초)':>10}   메모")
    print("-" * 100)
    for i, r in enumerate(RESULTS):
        delta = "—" if i == 0 else f"{r['val_acc'] - base:+.4f}"
        print(f"{i:<3}{r['name']:<28}{r['val_acc']:>9.4f}{delta:>10}"
              f"{r['epochs']:>7}{r['sec']:>10.0f}   {r['note']}")
    print(f"\n최고 기록 : {BEST['cfg']['name']}  val acc {BEST['acc']:.4f}")


def plot_results():
    if not RESULTS:
        return
    plt.figure(figsize=(10, 4))
    accs = [r["val_acc"] for r in RESULTS]
    bars = plt.bar(range(len(accs)), accs, color="#156082")
    bars[int(np.argmax(accs))].set_color("#E97132")
    for b, v in zip(bars, accs):
        plt.text(b.get_x() + b.get_width() / 2, v + 0.012, f"{v:.3f}",
                 ha="center", fontsize=11, fontweight="bold")
    plt.xticks(range(len(RESULTS)), [r["name"] for r in RESULTS],
               rotation=20, ha="right", fontsize=9)
    plt.ylabel("best validation accuracy"); plt.ylim(0, 1.05)
    plt.grid(axis="y", color="#EEEEEE"); plt.tight_layout(); plt.show()


show_results()
plot_results()

---
## 7. 최종 평가 — **마지막에 딱 한 번만**

지금까지의 모든 선택은 validation 으로만 했습니다.
이제 가장 좋았던 모델 하나를 test 셋으로 평가합니다.

> **이 셀을 실행한 뒤에는 모델을 더 고치지 마세요.**
> 고치고 다시 실행하면 그 순간 test 정확도는 의미를 잃습니다.

In [ ]:
criterion = nn.CrossEntropyLoss()
_, _, best_test_loader = BEST["loaders"]

test_loss, test_acc = evaluate(BEST["model"], best_test_loader, criterion)
print(f"최종 선택 모델 : {BEST['cfg']['name']}")
print(f"검증 정확도    : {BEST['acc']:.4f}")
print(f"테스트 정확도  : {test_acc:.4f}   <-- 제출 점수")

### 오류 분석 — 아직 무엇이 안 되는가

In [ ]:
@torch.no_grad()
def confusion_and_errors(model, loader):
    model.eval()
    cm = np.zeros((10, 10), dtype=int)
    for images, labels in loader:
        preds = model(images.to(device)).argmax(1).cpu().numpy()
        for t, p in zip(labels.numpy(), preds):
            cm[t, p] += 1
    return cm


cm = confusion_and_errors(BEST["model"], best_test_loader)
per_class = cm.diagonal() / cm.sum(axis=1)

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(cm / cm.sum(axis=1, keepdims=True), cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(10)); ax.set_yticks(range(10))
ax.set_xticklabels(CLASSES, rotation=45, ha="right"); ax.set_yticklabels(CLASSES)
ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title("Confusion matrix")
fig.colorbar(im, ax=ax, fraction=0.045); plt.tight_layout(); plt.show()

print("클래스별 정확도 (낮은 순)")
for i in np.argsort(per_class):
    print(f"  {CLASSES[i]:<12} {per_class[i]:.3f}")

print("\n가장 많이 헷갈린 쌍 TOP 5")
pairs = [(cm[i, j], i, j) for i in range(10) for j in range(10) if i != j]
for c, i, j in sorted(pairs, reverse=True)[:5]:
    print(f"  실제 {CLASSES[i]:<12} → {CLASSES[j]:<12} {c:4d} 장")

---
## 8. 결과 요약 — **제출물 ③**

아래 셀을 더블클릭해 다섯 항목을 채워 주세요.

**팀명 :**

**1. 최종 성능** (검증 / 테스트 정확도, 어떤 설정이었는지)

→

**2. 가장 효과가 컸던 변경** (무엇을, 몇 %p)

→

**3. 기대했지만 효과가 없었던 것** (무엇을 시도했고 왜 안 됐다고 생각하는지)

→

**4. 남은 실패 사례** (어떤 클래스에서 여전히 틀리는지, 왜 그럴 것 같은지)

→

**5. 시간이 더 있었다면 해 보고 싶은 것**

→

---

### 제출 전 체크리스트

- [ ] `런타임` → `모두 실행` 을 해도 오류 없이 끝난다
- [ ] 팀명·팀원을 맨 위에 적었다
- [ ] 섹션 6의 실험 기록표에 시도한 실험이 모두 남아 있다
- [ ] 섹션 7을 딱 한 번만 실행했다
- [ ] 섹션 8의 다섯 항목을 모두 채웠다
- [ ] 파일명을 `팀명_해커톤.ipynb` 로 바꿔 저장했다

수고하셨습니다.